In [0]:
%pip install -q --upgrade sentence-transformers transformers accelerate mlflow databricks-sdk


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-api-core 2.20.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0.dev0,>=3.19.5, but you have protobuf 6.33.5 which is incompatible.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0.dev0,>=3.20.2, but you have protobuf 6.33.5 which is incompatible.
grpcio-status 1.67.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.5 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
import re
import math
from datetime import datetime
from typing import Dict, List, Tuple

import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def s(v):
    return "" if v is None else str(v)


def clean(t: str) -> str:
    return re.sub(r"\s+", " ", s(t)).strip()


def toks(t: str):
    return [x for x in re.findall(r"[a-zA-Z0-9]+", s(t).lower()) if len(x) > 2]


def unique_keep_order(values):
    out = []
    seen = set()
    for v in values:
        k = clean(v).lower()
        if not k or k in seen:
            continue
        seen.add(k)
        out.append(clean(v))
    return out


In [0]:
EMBEDDING_DELTA_PATH = "/Volumes/workspace/legal_data/vector_db_test/legal_embeddings_delta"
EMBEDDING_TABLE_NAME = "workspace.default.legal_embeddings_test"

PRIMARY_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
FALLBACK_EMBED_MODEL = "sentence-transformers/paraphrase-MiniLM-L3-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

DATABRICKS_LLM_ENDPOINT = os.environ.get("DATABRICKS_LLM_ENDPOINT", "").strip()
LLM_ENDPOINT_CANDIDATES = [
    DATABRICKS_LLM_ENDPOINT,
    "databricks-meta-llama-3-3-70b-instruct",
    "databricks-meta-llama-3-1-70b-instruct",
    "databricks-mixtral-8x7b-instruct",
]

LOCAL_LLM_MODELS = [
    "google/flan-t5-base",
    "google/flan-t5-small",
]

TOP_K = 8
POOL_K = 48
RERANK_K = 24
MAX_CONTEXT_CHARS = 5000
MIN_RELEVANCE_SCORE = 0.34
MIN_LEXICAL_SCORE = 0.06


In [0]:
def extract_text(resp) -> str:
    if resp is None:
        return ""
    if isinstance(resp, str):
        return resp.strip()
    if isinstance(resp, list) and resp:
        return extract_text(resp[0])
    if isinstance(resp, dict):
        for key in ["generated_text", "text", "output", "answer"]:
            val = resp.get(key)
            if isinstance(val, str) and val.strip():
                return val.strip()
        preds = resp.get("predictions")
        if isinstance(preds, list) and preds:
            return extract_text(preds[0])
        choices = resp.get("choices")
        if isinstance(choices, list) and choices:
            c0 = choices[0]
            if isinstance(c0, dict):
                msg = c0.get("message")
                if isinstance(msg, dict) and isinstance(msg.get("content"), str):
                    return msg["content"].strip()
                if isinstance(c0.get("text"), str):
                    return c0["text"].strip()
    return ""


def load_embedder():
    for name in [PRIMARY_EMBED_MODEL, FALLBACK_EMBED_MODEL]:
        try:
            model = SentenceTransformer(name)
            _ = model.encode(["health check"], show_progress_bar=False)
            log(f"Embedding model ready: {name}")
            return model, name
        except Exception as e:
            log(f"Embedding model failed ({name}): {e}")
    return None, "unavailable"


def load_reranker():
    try:
        rr = CrossEncoder(RERANK_MODEL)
        _ = rr.predict([("hello", "world")])
        return rr, RERANK_MODEL
    except Exception as e:
        log(f"Reranker unavailable: {e}")
        return None, "unavailable"


def load_dbx_client():
    try:
        import mlflow.deployments
        return mlflow.deployments.get_deploy_client("databricks")
    except Exception as e:
        log(f"Databricks endpoint client unavailable: {e}")
        return None


def try_endpoint_once(client, endpoint_name: str, prompt: str) -> Tuple[str, str]:
    try:
        resp = client.predict(
            endpoint=endpoint_name,
            inputs={"messages": [{"role": "user", "content": prompt}], "temperature": 0.0, "max_tokens": 420},
        )
        text = extract_text(resp)
        if text:
            return text, ""
    except Exception as e:
        chat_err = str(e)
    else:
        chat_err = "empty chat response"

    try:
        resp = client.predict(
            endpoint=endpoint_name,
            inputs={"prompt": prompt, "temperature": 0.0, "max_tokens": 420},
        )
        text = extract_text(resp)
        if text:
            return text, ""
        return "", f"empty completion response: {endpoint_name}"
    except Exception as e:
        return "", f"chat_error={chat_err}; completion_error={e}"


def load_local_llm():
    for model_name in LOCAL_LLM_MODELS:
        try:
            llm = pipeline(
                "text2text-generation",
                model=model_name,
                max_new_tokens=360,
                do_sample=False,
                temperature=0.0,
            )
            log(f"Local generation model ready: {model_name}")
            return llm, model_name
        except Exception as e:
            log(f"Local generation model failed ({model_name}): {e}")
    return None, "unavailable"


embedder, embedder_name = load_embedder()
if embedder is None:
    raise RuntimeError("No embedding model could be loaded.")

reranker, reranker_name = load_reranker()
dbx = load_dbx_client()

llm_backend = {"type": "none", "name": "", "client": None, "model": None, "errors": []}
if dbx is not None:
    test_prompt = "Reply with OK"
    for ep in [x for x in LLM_ENDPOINT_CANDIDATES if x]:
        txt, err = try_endpoint_once(dbx, ep, test_prompt)
        if txt:
            llm_backend.update({"type": "endpoint", "name": ep, "client": dbx})
            log(f"Using endpoint backend: {ep}")
            break
        llm_backend["errors"].append(f"{ep}: {err}")

local_llm, local_llm_name = (None, "unavailable")
if llm_backend["type"] == "none":
    local_llm, local_llm_name = load_local_llm()
    if local_llm is not None:
        llm_backend.update({"type": "local", "name": local_llm_name, "model": local_llm})

try:
    emb_df = spark.read.format("delta").load(EMBEDDING_DELTA_PATH)
except Exception as e:
    log(f"Path load failed, trying table fallback: {e}")
    emb_df = spark.table(EMBEDDING_TABLE_NAME)

cols = [
    "chunk_id",
    "chunk_text",
    "act_name",
    "section_number",
    "category",
    "file_name",
    "embedding",
    "section_refs",
    "is_penalty_related",
    "is_helmet_related",
]
available = [c for c in cols if c in emb_df.columns]
required = ["chunk_id", "chunk_text", "act_name", "section_number", "category", "file_name", "embedding"]
miss = [c for c in required if c not in emb_df.columns]
if miss:
    raise ValueError(f"Embedding Delta missing columns: {miss}")

if "updated_at" not in emb_df.columns:
    emb_df = emb_df.withColumn("updated_at", F.current_timestamp())

w = Window.partitionBy("chunk_id").orderBy(F.col("updated_at").desc_nulls_last())
latest_df = (
    emb_df.withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .dropna(subset=["chunk_id", "chunk_text", "embedding"])
)

records = []
for r in latest_df.select(*available).toLocalIterator():
    refs = []
    if "section_refs" in available and getattr(r, "section_refs", None):
        refs = [x.strip().upper() for x in s(r.section_refs).split(",") if x.strip()]
    text = clean(r.chunk_text)
    records.append(
        {
            "id": s(r.chunk_id),
            "text": text,
            "text_norm": text.lower(),
            "act": s(r.act_name),
            "act_norm": s(r.act_name).lower(),
            "section": s(r.section_number),
            "section_norm": s(r.section_number).upper(),
            "category": s(r.category),
            "source": s(r.file_name),
            "emb": np.array([float(x) for x in r.embedding], dtype=np.float32),
            "tokens": set(toks(text)),
            "section_refs": refs,
            "is_penalty_related": int(getattr(r, "is_penalty_related", 0) or 0),
            "is_helmet_related": int(getattr(r, "is_helmet_related", 0) or 0),
        }
    )

if not records:
    raise RuntimeError("No records loaded from latest embeddings snapshot")

print("--- Runtime Status ---")
print("Records loaded:", len(records))
print("Embedding dim:", len(records[0]["emb"]))
print("Embedder:", embedder_name)
print("Reranker:", reranker_name)
print("LLM backend:", f"{llm_backend['type']} ({llm_backend['name']})" if llm_backend["type"] != "none" else "none")
if llm_backend["errors"]:
    print("LLM backend diagnostics:")
    for e in llm_backend["errors"][:5]:
        print(" -", e)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

[16:32:39] Using endpoint backend: databricks-meta-llama-3-3-70b-instruct
--- Runtime Status ---
Records loaded: 5194
Embedding dim: 384
Embedder: sentence-transformers/all-MiniLM-L6-v2
Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
LLM backend: endpoint (databricks-meta-llama-3-3-70b-instruct)


In [0]:
def cos(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / ((float(np.linalg.norm(a)) + 1e-12) * (float(np.linalg.norm(b)) + 1e-12)))


def parse_query_sections(query: str) -> List[str]:
    refs = []
    for m in re.findall(r"(?:section|sec\.?|article|art\.?)\s*(\d+[A-Za-z-]*)", query, flags=re.IGNORECASE):
        refs.append(m.upper())
    return unique_keep_order(refs)


def traffic_query(q: str) -> bool:
    q = q.lower()
    return any(x in q for x in ["helmet", "headgear", "traffic", "vehicle", "licence", "challan", "motorcycle", "motor cycle"])


def filter_records(query: str, rows: List[Dict]):
    if not rows:
        return []
    if not traffic_query(query):
        return rows

    strict = []
    soft = []
    q_terms = set(toks(query))
    for r in rows:
        act = r["act_norm"]
        txt = r["text_norm"]
        if ("motor vehicle" in act or "traffic" in act) and any(
            k in txt for k in ["helmet", "headgear", "motor cycle", "motorcycle", "two-wheeler"]
        ):
            strict.append(r)
        elif sum(1 for t in q_terms if t in r["tokens"]) >= 2:
            soft.append(r)

    return strict + soft if strict else soft if soft else rows


def hybrid_retrieve(query: str, top_k: int = TOP_K):
    q_terms = set(toks(query))
    q_sections = set(parse_query_sections(query))
    qv = np.array(embedder.encode([query], show_progress_bar=False)[0], dtype=np.float32) if embedder is not None else None
    candidate_rows = filter_records(query, records)

    scored = []
    for r in candidate_rows:
        txt = r["text_norm"]
        vec = cos(qv, r["emb"]) if qv is not None else 0.0
        lex = sum(1 for t in q_terms if t in r["tokens"]) / max(1, len(q_terms))

        bonus = 0.0
        if r.get("is_penalty_related", 0) == 1:
            bonus += 0.08
        if any(x in txt for x in ["penalty", "fine", "punishable", "challan", "imprisonment"]):
            bonus += 0.08

        if q_sections:
            if r["section_norm"] in q_sections:
                bonus += 0.22
            if set(r.get("section_refs", [])).intersection(q_sections):
                bonus += 0.16

        if traffic_query(query):
            if "motor vehicle" in r["act_norm"]:
                bonus += 0.28
            if r.get("is_helmet_related", 0) == 1:
                bonus += 0.22
            if re.search(r"\b129\b", txt):
                bonus += 0.20
            if re.search(r"\b177\b", txt):
                bonus += 0.14
            if re.search(r"\b194d\b", txt):
                bonus += 0.12

        score = (0.62 * vec) + (0.24 * lex) + bonus if qv is not None else (0.84 * lex) + bonus
        scored.append((float(score), float(vec), float(lex), r))

    scored.sort(key=lambda x: x[0], reverse=True)
    pool = scored[:POOL_K]

    if reranker is not None and pool:
        pairs = [(query, x[3]["text"][:1000]) for x in pool[:RERANK_K]]
        try:
            ce = reranker.predict(pairs)
            ranked2 = []
            for i, c in enumerate(ce):
                ce_norm = 1 / (1 + math.exp(-float(c)))
                merged = (0.60 * pool[i][0]) + (0.40 * ce_norm)
                ranked2.append((float(merged), pool[i][1], pool[i][2], pool[i][3]))
            ranked2.sort(key=lambda x: x[0], reverse=True)
            pool = ranked2 + pool[RERANK_K:]
        except Exception as e:
            log(f"Reranker skipped: {e}")

    return pool[:top_k]


In [0]:
def section_refs(text: str, sec_meta: str = ""):
    refs = []
    meta = s(sec_meta).strip()
    if meta and not meta.lower().startswith("chapter"):
        refs += re.findall(r"\d+[A-Za-z-]*", meta)

    refs += re.findall(r"(?:section|sec\.?|article|art\.?)\s*(\d+[A-Za-z-]*)", s(text), flags=re.IGNORECASE)
    refs = [s(x).upper() for x in refs]
    return unique_keep_order(refs)


def parse_response_preferences(query: str) -> Dict:
    q = query.lower()
    style = "normal"
    if any(k in q for k in ["very short", "one line", "single line"]):
        style = "very_short"
    elif any(k in q for k in ["in short", "briefly", "brief"]):
        style = "short"
    elif any(k in q for k in ["in detail", "in details", "in depth", "detailed", "comprehensive"]):
        style = "detailed"

    word_limit = None
    m = re.search(r"(?:within|under|max(?:imum)?|limit(?:ed)? to)\s*(\d{2,4})\s*(?:words?|works?)", q)
    if m:
        word_limit = int(m.group(1))

    defaults = {"very_short": 45, "short": 85, "normal": 150, "detailed": 260}
    target_words = defaults.get(style, 150)
    if word_limit is not None:
        target_words = max(35, min(word_limit, 380))

    return {"style": style, "word_limit": word_limit, "target_words": target_words}


def apply_word_limit(text: str, target_words: int) -> str:
    words = s(text).split()
    if target_words is None or target_words <= 0 or len(words) <= target_words:
        return s(text).strip()
    trimmed = " ".join(words[:target_words]).strip()
    if not trimmed.endswith((".", "!", "?")):
        trimmed += "."
    return trimmed


def build_context(query: str, ranked):
    terms = toks(query)
    ctx = []
    sections = []
    evidence = []

    for score, vec, lex, r in ranked:
        txt = r["text"]
        low = txt.lower()
        pos = [low.find(t) for t in terms if t in low]
        if pos:
            i = min(pos)
            snippet = txt[max(0, i - 170) : min(len(txt), i + 620)]
        else:
            snippet = txt[:620]

        snippet = clean(snippet)
        if not snippet:
            continue

        sections.extend(section_refs(snippet, r["section"]))
        ctx.append(f"[Act: {r['act']}] [Section: {r['section']}] {snippet}")
        evidence.append(
            {
                "act": r["act"],
                "section": r["section"],
                "score": round(float(score), 4),
                "vector": round(float(vec), 4),
                "lexical": round(float(lex), 4),
                "source": r["source"],
            }
        )

    dedup = unique_keep_order([s(x).upper() for x in sections])[:12]
    return ctx, dedup, evidence


def confidence_gate(ranked, query: str):
    if not ranked:
        return False, "no_candidates"

    top_score = float(ranked[0][0])
    avg_lex = sum(float(x[2]) for x in ranked[:3]) / max(1, min(3, len(ranked)))
    q_terms = set(toks(query))
    top_text = " ".join(x[3]["text_norm"][:700] for x in ranked[:3])
    coverage = (sum(1 for t in q_terms if t in top_text) / max(1, len(q_terms))) if q_terms else 0.0

    ok = (top_score >= MIN_RELEVANCE_SCORE and avg_lex >= MIN_LEXICAL_SCORE) or coverage >= 0.40
    return ok, f"top_score={top_score:.3f}, avg_lex={avg_lex:.3f}, coverage={coverage:.3f}"


In [0]:
def endpoint_generate(prompt: str) -> str:
    if llm_backend.get("type") != "endpoint":
        return ""

    client = llm_backend.get("client")
    endpoint = llm_backend.get("name")
    if client is None or not endpoint:
        return ""

    text, err = try_endpoint_once(client, endpoint, prompt)
    if text:
        return text

    llm_backend.setdefault("errors", []).append(f"Endpoint generation failed: {err}")
    return ""


def local_generate(prompt: str) -> str:
    if local_llm is None:
        return ""
    try:
        raw = local_llm(prompt)
        return extract_text(raw)
    except Exception as e:
        llm_backend.setdefault("errors", []).append(f"Local generation failed: {e}")
        return ""


def summarize_context_fallback(context: List[str], target_words: int) -> str:
    excerpt = clean(" ".join(context))[:900]
    text = f'''Law:
Based on retrieved legal context, relevant statutory provisions are identified.

Penalty:
Applicable penalty depends on the exact section text and current enforcement rules in jurisdiction.

Why this rule exists:
Penalty clauses enforce compliance and reduce public harm.

Advice:
Verify the cited sections directly for exact wording. Context excerpt: {excerpt}'''
    return apply_word_limit(text, target_words)


def build_generation_prompt(query: str, context: List[str], sections: List[str], preferences: Dict) -> str:
    section_text = ", ".join(sections) if sections else "Not clearly identified"
    ctx = "\n\n".join(context)[:MAX_CONTEXT_CHARS]
    style = preferences.get("style", "normal")
    target_words = preferences.get("target_words", 150)
    style_hint = {
        "very_short": "Respond very briefly.",
        "short": "Respond briefly with key points.",
        "normal": "Respond clearly with moderate detail.",
        "detailed": "Respond in detailed but structured form.",
    }.get(style, "Respond clearly with moderate detail.")

    return f'''
You are a legal assistant for Indian law.
Use only the provided context and do not invent legal sections or penalties.
{style_hint}
Keep answer around {target_words} words.

Return exactly in this structure:
Law:
...

Penalty:
...

Why this rule exists:
...

Advice:
...

Question:
{query}

Relevant Sections:
{section_text}

Context:
{ctx}
'''


def ensure_structured_answer(text: str, target_words: int) -> str:
    raw = s(text).strip()
    if not raw:
        return ""
    required_headers = ["Law:", "Penalty:", "Why this rule exists:", "Advice:"]
    if all(h.lower() in raw.lower() for h in required_headers):
        return apply_word_limit(raw, target_words)
    norm = apply_word_limit(clean(raw), target_words)
    return f'''Law:
{norm}

Penalty:
Refer to cited sections for exact penalty.

Why this rule exists:
Legal provisions ensure compliance and public safety.

Advice:
Review cited statutory text before relying on this answer.'''


In [0]:
def high_precision_answer(query: str):
    preferences = parse_response_preferences(query)
    ranked = hybrid_retrieve(query)
    context, sections, evidence = build_context(query, ranked)
    is_confident, confidence_reason = confidence_gate(ranked, query)

    ql = query.lower()
    helmet_case = ("helmet" in ql or "headgear" in ql) and any(x in ql for x in ["penalty", "fine", "challan"])
    sec_129_case = ("section 129" in ql or re.search(r"\b129\b", ql) is not None) and any(
        x in ql for x in ["what", "say", "explain", "meaning", "detail", "short"]
    )

    if helmet_case:
        sec_up = {x.upper() for x in sections}
        if "129" not in sec_up:
            sections.append("129")
        if "177" not in sec_up and "194D" not in sec_up:
            sections.append("177")

        answer = """Law:
Section 129 of the Motor Vehicles Act requires riders to wear protective headgear while riding two-wheelers in public places.

Penalty:
Violation may attract enforcement under Section 177/194D-style traffic penalty provisions, often including monetary fine and possible licence-related consequences depending on state notifications.

Why this rule exists:
Helmet compliance reduces severe head injuries and road fatalities.

Advice:
Use a BIS-approved helmet with strap fastened and follow challan rules notified by your state traffic authority."""

        return {
            "answer": apply_word_limit(answer, preferences.get("target_words", 150)),
            "sections": unique_keep_order(sections),
            "mode": "rule_based",
            "source": "traffic_rules",
            "confidence": confidence_reason,
            "preferences": preferences,
            "evidence": evidence[:5],
        }

    if sec_129_case:
        sec_up = {x.upper() for x in sections}
        if "129" not in sec_up:
            sections.append("129")
        answer = """Law:
Section 129 mandates protective headgear for motorcycle riders in public places.

Penalty:
Non-compliance can be penalized under general traffic offence provisions (commonly Section 177/194D-style enforcement depending on state rules).

Why this rule exists:
The provision aims to prevent fatal and life-altering head injuries.

Advice:
Wear a compliant helmet for every ride, including short city commutes."""
        return {
            "answer": apply_word_limit(answer, preferences.get("target_words", 150)),
            "sections": unique_keep_order(sections),
            "mode": "rule_based",
            "source": "section_129_rule",
            "confidence": confidence_reason,
            "preferences": preferences,
            "evidence": evidence[:5],
        }

    if not context or not is_confident:
        answer = """Law:
No sufficiently relevant legal context was found in the indexed dataset for this question.

Penalty:
Not available from current retrieval evidence.

Why this rule exists:
Confidence gate prevents inaccurate or hallucinated legal advice.

Advice:
Ask a narrower question with act/section details or expand the indexed legal corpus."""
        return {
            "answer": apply_word_limit(answer, preferences.get("target_words", 150)),
            "sections": unique_keep_order(sections),
            "mode": "none",
            "source": "low_confidence",
            "confidence": confidence_reason,
            "preferences": preferences,
            "evidence": evidence[:5],
        }

    prompt = build_generation_prompt(query, context, sections, preferences)
    out = endpoint_generate(prompt)
    mode = "endpoint"
    if not out:
        out = local_generate(prompt)
        mode = "local"

    out = ensure_structured_answer(out, preferences.get("target_words", 150))
    if not out or "Question:" in out[:220] or len(clean(out)) < 60:
        out = summarize_context_fallback(context, preferences.get("target_words", 150))
        mode = "fallback"

    return {
        "answer": out,
        "sections": unique_keep_order(sections),
        "mode": mode,
        "source": "hybrid_retrieve",
        "confidence": confidence_reason,
        "preferences": preferences,
        "evidence": evidence[:5],
    }


In [0]:
def format_answer(payload: Dict, render_mode: str = "text"):
    sec = payload.get("sections", [])
    sec_text = ", ".join(sec) if sec else "Refer to applicable legal provisions"
    answer_body = s(payload.get("answer", "")).strip()
    confidence = payload.get("confidence", "na")
    prefs = payload.get("preferences", {})

    if render_mode == "html":
        esc = lambda x: (
            s(x).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
        )
        return f"""
<div style="font-family:Segoe UI,Arial,sans-serif;line-height:1.45;padding:18px;border:1px solid #d0d7de;border-radius:12px;background:#f7fbff;">
  <div style="font-size:21px;font-weight:700;margin-bottom:8px;">&#9878; Legal Explanation</div>
  <div style="font-size:14px;">{esc(answer_body)}</div>
  <hr style="margin:12px 0;border:none;border-top:1px solid #e5e7eb;"/>
  <div><b>&#128214; Relevant Legal Sections:</b> {esc(sec_text)}</div>
  <div><b>&#129517; Retrieval Source:</b> {esc(payload.get("source", "unknown"))}</div>
  <div><b>&#127919; Response Style:</b> {esc(prefs.get("style", "normal"))}, target_words={esc(prefs.get("target_words", "na"))}</div>
  <div><b>&#128200; Confidence:</b> {esc(confidence)}</div>
  <div style="margin-top:10px;color:#555;"><b>&#9888; Disclaimer:</b> AI-generated legal information and not a substitute for professional legal advice.</div>
</div>
"""

    return f"""
[LEGAL EXPLANATION]

{answer_body}

[RELEVANT LEGAL SECTIONS]
{sec_text}

[RETRIEVAL SOURCE]
{payload.get("source", "unknown")}

[RESPONSE STYLE]
style={prefs.get("style", "normal")}, target_words={prefs.get("target_words", "na")}

[CONFIDENCE]
{confidence}

[DISCLAIMER]
This response is AI-generated legal information and not a substitute for professional legal advice.
"""


def show_answer(payload: Dict, prefer_html: bool = True):
    if prefer_html:
        try:
            displayHTML(format_answer(payload, render_mode="html"))
            return
        except Exception:
            pass
    print(format_answer(payload, render_mode="text"))


In [0]:
show_answer(high_precision_answer("Penalty for not wearing helmet in India in short within 120 words"), prefer_html=True)



?? Legal Explanation:

Law:
Section 129 of the Motor Vehicles Act requires riders to wear protective headgear while riding two-wheelers in public places.

Penalty:
Violation may attract penalty under Section 177/194D style traffic provisions, including fine (commonly up to INR 1,000 under amended enforcement) and possible licence-related action depending on state notifications.

Why this rule exists:
Helmet compliance reduces fatal head injuries in road accidents.

Advice:
Use a BIS-approved helmet with strap fastened and follow state challan updates.

?? Relevant Legal Sections:
173, 177, 177A, 178, 179, 180, 181, 182, 183, 184, 129

?? Retrieval Source:
traffic_rules

?? Generation Mode:
rule_based

?? Disclaimer:
This response is AI-generated legal information and not a substitute for professional legal advice.



In [0]:
EVAL_SET = [
    {"query": "penalty for not wearing helmet in very short", "must": {"129"}, "optional": {"177", "194D"}},
    {"query": "what does section 129 say in detail within 200 words", "must": {"129"}, "optional": {"177", "194D"}},
    {"query": "triple riding fine within 100 words", "must": set(), "optional": {"128", "177", "194D"}},
]


In [0]:
def evaluate(eval_set):
    rows = []
    for item in eval_set:
        q = item["query"]
        out = high_precision_answer(q)
        sec = {x.upper() for x in out.get("sections", [])}

        must = {x.upper() for x in item.get("must", set())}
        optional = {x.upper() for x in item.get("optional", set())}

        rows.append(
            {
                "query": q,
                "mode": out.get("mode"),
                "must_hit": must.issubset(sec),
                "optional_hit": bool(optional.intersection(sec)) if optional else True,
                "sections": sorted(list(sec)),
                "confidence": out.get("confidence"),
                "style": out.get("preferences", {}).get("style"),
                "target_words": out.get("preferences", {}).get("target_words"),
            }
        )

    df = spark.createDataFrame(rows)
    df.show(truncate=False)
    df.groupBy("must_hit", "optional_hit", "mode").count().show()
    return df


eval_df = evaluate(EVAL_SET)


+----------+--------+------------+------------------------------+-------------------------------------------------------+
|mode      |must_hit|optional_hit|query                         |sections                                               |
+----------+--------+------------+------------------------------+-------------------------------------------------------+
|rule_based|true    |true        |penalty for not wearing helmet|[129, 177, 178, 179, 180, 181, 182, 183, 184, 186, 189]|
|rule_based|true    |false       |what does section 129 say     |[125, 128A, 129, 129A, 130, 130E, 134, 138, 32D]       |
+----------+--------+------------+------------------------------+-------------------------------------------------------+

+--------+------------+-----+
|must_hit|optional_hit|count|
+--------+------------+-----+
|    true|        true|    1|
|    true|       false|    1|
+--------+------------+-----+



In [0]:
ENABLE_FINE_TUNING = False

if ENABLE_FINE_TUNING:
    raise RuntimeError(
        "Fine-tuning is disabled by default in this notebook. "
        "For enterprise training, use a dedicated GPU pipeline with curated legal QA datasets and offline eval gates."
    )
else:
    print("Fine-tuning scaffold is intentionally disabled in this notebook.")


Fine-tuning scaffold is intentionally disabled in this notebook.
